<a href="https://colab.research.google.com/github/cesarprando/am_turma5_prova/blob/main/ProvaIcoma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

In [ ]:
from google.colab import files

arquivos = files.upload()

df_doc = pd.read_csv(
    'Tabela_Docente_2025_V2.csv',
    sep=';',
    encoding='latin1',
    low_memory=False
)

df_escola = pd.read_csv(
    'Tabela_Escola_2025_V2.csv',
    sep=';',
    encoding='latin1',
    low_memory=False
)

print("Tabela de docentes:", df_doc.shape)
print("Tabela de escolas:", df_escola.shape)

display(df_doc.head())
display(df_escola.head())

In [ ]:
colunas_docentes = [
    'CO_ENTIDADE',
    'QT_DOC_BAS_DOCENTE',
    'QT_DOC_BAS_ESCO_SUP_GRAD'
]

doc_qualif = df_doc[colunas_docentes].copy()


for coluna in colunas_docentes[1:]:
    doc_qualif[coluna] = pd.to_numeric(
        doc_qualif[coluna], errors='coerce'
    )


doc_qualif = doc_qualif.dropna(
    subset=['CO_ENTIDADE', 'QT_DOC_BAS_DOCENTE',
            'QT_DOC_BAS_ESCO_SUP_GRAD']
)

doc_qualif = doc_qualif[
    doc_qualif['QT_DOC_BAS_DOCENTE'] > 0
].copy()


doc_qualif['pct_superior'] = (
    doc_qualif['QT_DOC_BAS_ESCO_SUP_GRAD']
    / doc_qualif['QT_DOC_BAS_DOCENTE']
) * 100

doc_qualif = doc_qualif[
    ['CO_ENTIDADE', 'pct_superior']
].drop_duplicates(subset='CO_ENTIDADE')

print("Percentual calculado por escola:")
display(doc_qualif.head())

print("Quantidade de escolas com dados de qualificação:",
      len(doc_qualif))


In [ ]:

df_escola_modelo = df_escola[
    ['CO_ENTIDADE', 'TP_DEPENDENCIA', 'TP_LOCALIZACAO']
].copy()


df_p6 = pd.merge(
    df_escola_modelo,
    doc_qualif,
    on='CO_ENTIDADE',
    how='inner'
)


df_p6 = df_p6.dropna(
    subset=[
        'TP_DEPENDENCIA',
        'TP_LOCALIZACAO',
        'pct_superior'
    ]
)


df_p6 = df_p6.drop_duplicates(
    subset='CO_ENTIDADE'
)

df_p6 = df_p6.sample(
    n=min(5000, len(df_p6)),
    random_state=42
)


df_p6['target_alta_qualificacao'] = (
    df_p6['pct_superior'] > 85
).astype(int)

print("Quantidade de escolas utilizadas:", len(df_p6))
print("\nDistribuição da variável-alvo:")
print(df_p6['target_alta_qualificacao'].value_counts())

print("\nPercentual de cada classe:")
print(
    (df_p6['target_alta_qualificacao']
     .value_counts(normalize=True) * 100).round(2)
)


In [ ]:
X_p6 = df_p6[
    ['TP_DEPENDENCIA', 'TP_LOCALIZACAO']
].astype(str)


X_p6 = pd.get_dummies(
    X_p6,
    columns=['TP_DEPENDENCIA', 'TP_LOCALIZACAO'],
    dtype=int
)


y_p6 = df_p6['target_alta_qualificacao']

print("Formato das variáveis explicativas:", X_p6.shape)
print("\nVariáveis criadas pelo One-Hot Encoding:")
print(X_p6.columns.tolist())

display(X_p6.head())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_p6,
    y_p6,
    test_size=0.25,
    random_state=42,
    stratify=y_p6
)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Quantidade de registros para treino:", len(X_train))
print("Quantidade de registros para teste:", len(X_test))
print("Padronização concluída!")

In [ ]:
gnb_p6 = GaussianNB()

gnb_p6.fit(X_train_scaled, y_train)


y_pred = gnb_p6.predict(X_test_scaled)

print("Modelo GaussianNB treinado com sucesso!")

In [ ]:
n_classes = y_train.nunique()

if n_classes == 2 and y_train.value_counts().min() >= 5:
    scores_cv = cross_val_score(
        GaussianNB(),
        X_train_scaled,
        y_train,
        cv=5,
        scoring='accuracy'
    )

    print("=== VALIDAÇÃO CRUZADA ===")
    print("Acurácias por divisão:", scores_cv.round(4))
    print(f"Média da acurácia: {scores_cv.mean():.4f}")
    print(f"Desvio padrão: {scores_cv.std():.4f}")
else:
    print(
        "Não foi possível realizar validação cruzada "
        "com 5 divisões. Verifique a distribuição das classes."
    )


print("\n=== RESULTADOS DA PROVA PRÁTICA 6 ===")

print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}")

print(
    f"Precisão: "
    f"{precision_score(y_test, y_pred, zero_division=0):.4f}"
)

print(
    f"Recall: "
    f"{recall_score(y_test, y_pred, zero_division=0):.4f}"
)

print("\n=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=[
            'Qualificação não alta',
            'Alta qualificação'
        ],
        zero_division=0
    )
)

